# Evaluation on the attack graph: empirical exploitability (Table 2, AG column)
Every frozen defender plays the same 200 pre-drawn episodes (wiring, attacker type, $\lambda$, and initial state) against four admissible attackers:

1. `Trained P2`, the trained physical attacker,
2. `Critic-BR P2`, a myopic best response to the defender's current mixed action under the critic,
3. `Equilibrium P2`, the equilibrium column of the critic's stage game, and
4. `BR-PPO P2`, a recurrent-PPO best response trained from scratch against that defender (`best_response.ipynb`).

The first three act uniformly at random with probability $\lambda$; the fourth applies the same mixing internally. Returns are the defender's discounted payoff, so attackers minimize them. A defender's worst-case return $\widehat W$ is its lowest mean over the attackers, and its empirical exploitability is $\widehat\Delta = \widehat V^\star - \widehat W$, where $\widehat V^\star$ is the best worst-case return among the evaluated defenders. Every defender samples its action from its own distribution, as at deployment.

In [ ]:
%run Initialize.ipynb

## Settings and trained models

In [ ]:
from torch.distributions import Categorical

episode_length = 50
GAMMA = 0.99
num_episodes = 200
SAVE_DIR = "compared"

Context_Model = Load(TSTransformer().to(device), "encoder.pth")
critic        = Load(Critic().to(device), "critic.pth")
Player1       = Load(DefenderActor().to(device), "player1.pth")
Player2       = Load(AttackerActor().to(device), "player2.pth")
BASELINES     = Load_Baselines(SAVE_DIR)

# Pre-drawn episode configurations shared by every defender-attacker pair
episode_configs = []
cfg_rng = np.random.RandomState(12345)
for _ in range(num_episodes):
    p = cfg_rng.permutation(TERMINALS)
    s1_parent, s2_parent = int(p[0]), int(p[1])
    t0 = cfg_rng.randint(0, 2)
    t1 = cfg_rng.randint(1 - t0, 2)
    lam = cfg_rng.uniform(0.0, 0.5)
    x0 = cfg_rng.randint(n)
    episode_configs.append((s1_parent, s2_parent, t0, t1, lam, x0))

## Attackers (`BRAttacker` is defined in `Initialize.ipynb`)

In [ ]:
BR_CACHE = {}
def _load_br(def_name):
    if def_name not in BR_CACHE:
        ck = torch.load(os.path.join(SAVE_DIR, "BR_attacker_" + def_name.replace(" ", "_") + ".pth"),
                        map_location=device, weights_only=True)
        att = BRAttacker(rnn_h=ck["config"]["RNN_H"]).to(device)
        att.load_state_dict(ck["attacker"]); att.eval()
        BR_CACHE[def_name] = att
    return BR_CACHE[def_name]


class BRAttackerWrapper:
    def __init__(self, def_name):
        self.att = _load_br(def_name)
    def reset(self, Theta, lam):
        self.h = None
        self.u_prev = np.zeros(n_u, dtype=np.float32)
        self.theta_t = torch.tensor(Theta, dtype=torch.float32, device=device).view(1, 1, Len_Theta)
        self.lam = lam
    def act(self, x_bits):
        obs = np.concatenate([x_bits.astype(np.float32), self.u_prev])
        with torch.no_grad():
            logits, _, self.h = self.att(torch.tensor(obs, device=device).view(1, 1, d + n_u), self.theta_t, self.h)
        u_base = int(Categorical(logits=logits.view(-1)).sample())
        u = int(np.random.randint(n_u)) if np.random.rand() < self.lam else u_base
        self.u_prev = np.zeros(n_u, dtype=np.float32); self.u_prev[u_base] = 1.0
        return u


def attacker_action(Attacker, a_dist, lam, critic_Q, x_t, m2_t, tt_t):
    # critic_Q() returns the defender's payoff matrix under the critic, computed only when needed
    if np.random.rand() < lam:
        return int(np.random.choice(n_u))
    if Attacker == 'Trained P2':
        with torch.no_grad():
            u_dist = Player2(x_t, m2_t, tt_t).cpu().numpy()
        u_dist = u_dist / u_dist.sum()
        return int(np.random.choice(n_u, p=u_dist))
    Q = critic_Q()
    if Attacker == 'Critic-BR P2':
        return int(np.argmin(a_dist @ Q))            # minimizes the defender's expected payoff
    if Attacker == 'Equilibrium P2':
        _, _, q = MatrixValue(Q)
        q = np.clip(q, 0, None); q = q / q.sum()
        return int(np.random.choice(n_u, p=q))
    raise ValueError(Attacker)

## Rollout

In [ ]:
def run_episodes(Defender, Attacker):
    defw = get_defender(Defender)
    br = BRAttackerWrapper(Defender) if Attacker == 'BR-PPO P2' else None
    ep_disc = []
    for i_episode in tqdm(range(num_episodes), desc=f"{Defender} vs {Attacker}", leave=False):
        np.random.seed(1000 + i_episode); torch.manual_seed(1000 + i_episode)
        random.seed(1000 + i_episode)                      # Next_State draws from Python's random
        s1_parent, s2_parent, t0, t1, lam, x0 = episode_configs[i_episode]
        Theta, Type_Theta, D, rho = Context_From_Config(s1_parent, s2_parent, t0, t1, lam)
        defw.reset(Theta, Type_Theta, lam, x0, D, rho)
        if br is not None:
            br.reset(Theta, lam)
        x = x0
        # The critic's view of the episode, used by the critic-based attackers
        z_s = np.ones(d) / 2
        Context = np.zeros([Context_length, d * 3], dtype=np.float32)
        m2 = np.full(d, 0.2)
        counts = np.zeros(d)
        th_t = torch.tensor(Theta, dtype=torch.float32, device=device)
        tt_t = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        disc_sum = 0.0
        for t in range(episode_length):
            x_bits = State_Matrix[x, :]
            x_t = torch.tensor(x_bits, dtype=torch.float32, device=device)
            m2_t = torch.tensor(m2, dtype=torch.float32, device=device)

            def critic_Q():
                ctx = torch.tensor(Context, dtype=torch.float32, device=device).view(1, Context_length, d * 3)
                with torch.no_grad():
                    Emb = Context_Model(ctx, Context_Mask(t)).squeeze(0)
                    z_s_t = torch.tensor(z_s, dtype=torch.float32, device=device)
                    return critic(z_s_t, Emb, x_t, m2_t, tt_t, th_t).cpu().numpy().reshape(n_a, n_u)

            a_dist = defw.act_dist(x_bits, m2)
            u = br.act(x_bits) if br is not None else attacker_action(Attacker, a_dist, lam, critic_Q, x_t, m2_t, tt_t)
            a = int(np.random.choice(n_a, p=a_dist))       # the defender samples from its own policy
            r = Reward(x, a, u, Theta, D, rho)
            next_x = Next_State(x, a, u, D, rho)
            disc_sum += (GAMMA ** t) * r
            next_bits = State_Matrix[next_x, :]
            next_m2, counts = m2_update(x_bits, next_bits, m2, counts)
            y = Observation(next_bits)
            defw.observe(a, x_bits, next_bits, y)
            next_z_s = Def_State_Blf(z_s, y, A[a])
            row = Context_Row(z_s, a, next_z_s)
            if t < Context_length:
                Context[t, :] = row
            else:
                Context[:-1, :] = Context[1:, :]
                Context[-1, :] = row
            x, z_s, m2 = next_x, next_z_s, next_m2
        ep_disc.append(disc_sum)
    return np.asarray(ep_disc)

## Evaluate every defender against every attacker

In [ ]:
ATTACKERS = ['Trained P2', 'Critic-BR P2', 'Equilibrium P2', 'BR-PPO P2']
DEFENDERS = ['Critic P1', 'Trained P1', 'RPPO', 'DR_RPPO', 'VARIBAD', 'PEARL', 'MMD', 'Random P1']

all_runs = {}
for Defender in DEFENDERS:
    all_runs[Defender] = {Attacker: run_episodes(Defender, Attacker) for Attacker in ATTACKERS}
    print(Defender, {k: round(v.mean(), 3) for k, v in all_runs[Defender].items()})

## Worst-case return and empirical exploitability
`W / W_Oracle` is the fraction of the Oracle's worst-case return each defender recovers, the metric of the encoder ablation (Table 3).

In [ ]:
worst_runs = {}
for Defender in DEFENDERS:
    runs = all_runs[Defender]
    worst = min(runs, key=lambda k: runs[k].mean())          # attackers minimize the defender's return
    worst_runs[Defender] = (worst, runs[worst])

anchor = max(worst_runs, key=lambda k: worst_runs[k][1].mean())
V_star = worst_runs[anchor][1]
W_oracle = worst_runs['Critic P1'][1].mean()

print(f"anchor defender (least exploitable): {anchor}")
print(f"{'Defender':<12}{'worst attacker':>16}{'W':>9}{'Delta':>9}{'SE':>8}{'W / W_Oracle':>14}")
print("-" * 68)
for Defender in DEFENDERS:
    worst, W = worst_runs[Defender]
    Delta = V_star - W                                         # paired over the shared episodes
    print(f"{Defender:<12}{worst:>16}{W.mean():>9.3f}{Delta.mean():>9.3f}{Delta.std(ddof=1) / np.sqrt(len(Delta)):>8.3f}"
          f"{W.mean() / W_oracle:>13.0%}")